# 02. 課題仮説とベースラインモデル比較
**目的:** 入電数変動の仮説を整理したうえで、統計的アプローチ（SARIMA）と機械学習アプローチ（Prophet, XGBoost）の
ベースラインモデルを比較し、以降の改善のベースラインを定める。

**このNotebookでわかること:**
- 入電数変動に関する仮説
- 入電データの定常性（ADF検定）
- SARIMA / Prophet / XGBoostの予測精度比較とXGBoostが優位である根拠


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

from src.data.loader import PROCESSED_DATA_DIR
from src.models.forecasting import adf_test, evaluate_forecast, fit_sarima_forecast, run_xgboost_pipeline

## 課題仮説立案

- 入電数は曜日・休日といったカレンダー要因に強く依存する
- CM放映や検索需要、アカウント開設数の増加は数日〜1週間程度のラグを伴って入電数に波及する
- 税制改正や大規模障害など突発的なイベントが短期的なスパイクを生む
- これらの要因を特徴量として組み込むことで、XGBoostのような非線形モデルは
  周期性しか捉えられない統計モデルより高い予測精度を達成できるはずである

In [ ]:
regi_call_df = pd.read_csv(PROCESSED_DATA_DIR / 'regi_call.csv', parse_dates=['cdr_date'])

train = regi_call_df[regi_call_df['cdr_date'] < '2020-01-01']
test = regi_call_df[regi_call_df['cdr_date'] >= '2020-01-01']

## 統計学的アプローチ

### 入電データの定常性確認
そのままのデータで統計学的手法（ARIMA系）を使えるのか、ADF検定で確認する。

In [ ]:
adf_test(regi_call_df['call_num'])

p値が高く、定常であるとは言えない。1階差分を取って再検定する。

In [ ]:
regi_call_df['call_num_diff1'] = regi_call_df['call_num'].diff()
adf_test(regi_call_df['call_num_diff1'].dropna())

1階差分を取ることで定常性が確認できたため、SARIMAではこの階差構造をorderのd=1として利用する。

### SARIMAモデルを用いた予測

In [ ]:
forecast_sarima = fit_sarima_forecast(train, test, order=(6, 1, 6), seasonal_order=(1, 0, 1, 7))

plt.figure(figsize=(12, 5))
plt.plot(test['cdr_date'], test['call_num'], label='実測値', marker='o')
plt.plot(test['cdr_date'], forecast_sarima, label='予測値（SARIMA）', marker='o')
plt.title('SARIMA(6,1,6)(1,0,1,7) による入電数予測')
plt.xlabel('日付')
plt.ylabel('入電数')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

sarima_scores = evaluate_forecast(test['call_num'], forecast_sarima)

## 機械学習的アプローチ

### Prophetによる予測モデル

In [ ]:
prop_train = train[['cdr_date', 'call_num']].rename(columns={'cdr_date': 'ds', 'call_num': 'y'})
prop_test = test[['cdr_date', 'call_num']].rename(columns={'cdr_date': 'ds', 'call_num': 'y'})
prop_train['holiday_flag'] = train['holiday_flag']

model_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True, changepoint_prior_scale=0.5)
model_prophet.add_regressor('holiday_flag')
model_prophet.fit(prop_train)

future = model_prophet.make_future_dataframe(periods=len(prop_test), freq='D')
future['holiday_flag'] = pd.concat([train['holiday_flag'], test['holiday_flag']]).reset_index(drop=True)

forecast = model_prophet.predict(future)
forecast_test = forecast[forecast['ds'].isin(prop_test['ds'])]

In [ ]:
prophet_scores = evaluate_forecast(prop_test['y'], forecast_test['yhat'])

plt.figure(figsize=(12, 5))
plt.plot(prop_test['ds'], prop_test['y'], label='実測値', marker='o')
plt.plot(prop_test['ds'], forecast_test['yhat'], label='予測値（Prophet）', marker='o')
plt.title('Prophet による入電数予測')
plt.xlabel('日付')
plt.ylabel('入電数')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### XGBoostによる予測モデル

In [ ]:
train = train.copy()
test = test.copy()
train['week'] = train['cdr_date'].dt.isocalendar().week
train['month'] = train['cdr_date'].dt.month
test['week'] = test['cdr_date'].dt.isocalendar().week
test['month'] = test['cdr_date'].dt.month

xgb_model, X_train, y_train, X_test, y_test, y_pred = run_xgboost_pipeline(train, test)
xgb_scores = evaluate_forecast(y_test, y_pred)

### 各モデルを比較

In [ ]:
scores = pd.DataFrame({
    'Model': ['SARIMA', 'Prophet', 'XGBoost'],
    'MAE': [sarima_scores['MAE'], prophet_scores['MAE'], xgb_scores['MAE']],
    'RMSE': [sarima_scores['RMSE'], prophet_scores['RMSE'], xgb_scores['RMSE']],
    'MAPE': [sarima_scores['MAPE'], prophet_scores['MAPE'], xgb_scores['MAPE']],
})
scores

In [ ]:
y_pos = np.arange(len(scores))
bar_width = 0.4

fig, ax = plt.subplots(figsize=(12, 4))
ax.barh(y_pos - bar_width / 2, scores['MAE'], height=bar_width, label='MAE', color='skyblue')
ax.barh(y_pos + bar_width / 2, scores['RMSE'], height=bar_width, label='RMSE', color='salmon')
ax.set_yticks(y_pos)
ax.set_yticklabels(scores['Model'])
ax.invert_yaxis()
ax.set_xlabel('スコア')
ax.set_title('モデル別のMAE・RMSE比較')
ax.legend()
plt.tight_layout()
plt.show()

**XGBoostが3モデルの中で最も評価指標が良く、以降はXGBoostをベースに特徴量エンジニアリングと
ハイパーパラメータチューニングでスコア向上を目指す（次のNotebookに続く）。**